# M5 Forecasting: Model Performance and Statistical Analysis

In this notebook, we analyze the rolling backtest validation results, compare our LightGBM/CatBoost models against baselines, and evaluate the final champion model on the untouched test horizon.

In [ ]:
import json
import os
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Load Validation metrics
metrics_pkl = '../reports/val_metrics.pkl'
if os.path.exists(metrics_pkl):
    with open(metrics_pkl, 'rb') as f:
        val_metrics = pickle.load(f)

    # Convert to DataFrame
    summary_data = []
    for model_name, folds in val_metrics.items():
        for fold_idx, fold_data in enumerate(folds):
            summary_data.append({
                'Model': model_name,
                'Fold': fold_idx + 1,
                'WMAPE': fold_data['wmape'],
                'RMSSE': fold_data['rmsse']
            })
    df_val = pd.DataFrame(summary_data)
    display(df_val.groupby('Model')[['WMAPE', 'RMSSE']].mean().reset_index())
else:
    print("Validation metrics file not found!")

## 2. Compare Validation Error Distributions

In [ ]:
if os.path.exists(metrics_pkl):
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_val, x='Model', y='RMSSE', errorbar='sd', palette='Set2')
    plt.title('Mean Validation RMSSE across Folds (with Std Dev)')
    plt.ylabel('RMSSE')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

## 3. Load Statistical and Test Metrics

In [ ]:
metrics_json_path = '../reports/metrics.json'
if os.path.exists(metrics_json_path):
    with open(metrics_json_path, 'r') as f:
        test_metrics = json.load(f)
    print("=== Final Test Horizon Results ===")
    print(json.dumps(test_metrics, indent=4))
else:
    print("Test metrics file not found at reports/metrics.json!")

## 4. Visualizing Actuals vs. Predictions on Test Horizon

In [ ]:
forecast_parquet_path = '../data/processed/final_test_forecast.parquet'
if os.path.exists(forecast_parquet_path):
    df_forecast = pd.read_parquet(forecast_parquet_path)
    df_forecast['date'] = pd.to_datetime(df_forecast['date'])

    # Select a few active series to plot
    active_series = df_forecast.groupby(['item_id', 'store_id'])['demand'].sum().reset_index()
    active_series = active_series.sort_values(by='demand', ascending=False).head(3)

    for _, row in active_series.iterrows():
        item_id = row['item_id']
        store_id = row['store_id']

        sub_df = df_forecast[(df_forecast['item_id'] == item_id) & (df_forecast['store_id'] == store_id)].sort_values('date')

        plt.figure(figsize=(12, 4))
        plt.plot(sub_df['date'], sub_df['demand'], label='Actual Demand', color='black', marker='o', alpha=0.8)
        plt.plot(sub_df['date'], sub_df['forecast'], label='LightGBM Forecast', color='orange', linestyle='--', marker='x')
        plt.title(f'Actual vs Forecast for {item_id} at Store {store_id}')
        plt.xlabel('Date')
        plt.ylabel('Demand Units')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
else:
    print("Test forecast parquet file not found!")